In [1]:
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold, StratifiedShuffleSplit
import xgboost as xgb
from datetime import datetime
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import roc_auc_score

Model 1:

In [2]:
def objective(trial, features, target):
    scale_pos_weight = sum(1-target)/sum(target)
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.1, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight': scale_pos_weight,
        'n_estimators': 250,
        'objective': 'binary:logistic',
        'random_state': 20240627,
        'eval_metric': 'auc'
    }
    n_splits = 5
    seed = 20240627
    model = xgb.XGBClassifier(**params)
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = cross_val_score(model, features, target, cv=cv, scoring='roc_auc')
    return auc_scores.mean()

In [3]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_1"
    model_category = "xgboost"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 100 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=1985))
    study.optimize(lambda trial: objective(trial, X_data, y_data), n_trials=100, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 02:55:34,825] A new study created in memory with name: model_1_tuning
[I 2026-05-11 02:55:47,015] Trial 18 finished with value: 0.8986038096621753 and parameters: {'learning_rate': 0.22673406582169134, 'max_depth': 2, 'min_child_weight': 4, 'subsample': 0.640443222411407, 'colsample_bytree': 0.6000108550758412, 'gamma': 0.0003562726192535928, 'reg_lambda': 4.130813359413736e-06}. Best is trial 18 with value: 0.8986038096621753.
[I 2026-05-11 02:55:47,536] Trial 16 finished with value: 0.8961896650323012 and parameters: {'learning_rate': 0.8410627932818435, 'max_depth': 2, 'min_child_weight': 3, 'subsample': 0.6554916332550549, 'colsample_bytree': 0.8223423990696384, 'gamma': 7.696417897698708e-08, 'reg_lambda': 0.00016598655648055897}. Best is trial 18 with value: 0.8986038096621753.
[I 2026-05-11 02:55:47,540] Trial 2 finished with value: 0.8975935096016728 and parameters: {'learning_rate': 0.886541964682287, 'max_depth': 2, 'min_child_weight': 7, 'subsample': 0.73471699

Time: 0:02:49.313642


Model 2:

In [4]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_2"
    model_category = "xgboost"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 100 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=1985))
    study.optimize(lambda trial: objective(trial, X_data, y_data), n_trials=100, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 02:58:24,159] A new study created in memory with name: model_2_tuning
[I 2026-05-11 02:58:36,747] Trial 1 finished with value: 0.8251513885354876 and parameters: {'learning_rate': 0.34760290972151386, 'max_depth': 2, 'min_child_weight': 10, 'subsample': 0.8447052822271535, 'colsample_bytree': 0.750202358150595, 'gamma': 3.354069282534353e-05, 'reg_lambda': 1.929085363433942e-06}. Best is trial 1 with value: 0.8251513885354876.
[I 2026-05-11 02:58:36,965] Trial 7 finished with value: 0.7953379200868769 and parameters: {'learning_rate': 0.9466320142257222, 'max_depth': 2, 'min_child_weight': 5, 'subsample': 0.2851829855848422, 'colsample_bytree': 0.7602284836870272, 'gamma': 1.623050441411005e-08, 'reg_lambda': 7.010163057454475e-06}. Best is trial 1 with value: 0.8251513885354876.
[I 2026-05-11 02:58:39,467] Trial 2 finished with value: 0.8118709700507651 and parameters: {'learning_rate': 0.7926776907612112, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.6499304431

Time: 0:02:41.452635


Model 1a:

In [5]:
def objective_a(trial, features, target):
    scale_pos_weight = sum(1-target)/sum(target)
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.1, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight': scale_pos_weight,
        'n_estimators': 250,
        'objective': 'binary:logistic',
        'random_state': 20240627,
        'eval_metric': 'auc'
    }
    n_splits = 5
    seed = 20240627
    model = xgb.XGBClassifier(**params)
    cv = StratifiedShuffleSplit(n_splits=n_splits,
                                test_size=1/n_splits,
                                random_state=seed)
    auc_scores = []
    # cross validation loop for oversample
    for train_idx, val_idx in cv.split(features, target):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        # oversample
        oversampler = RandomOverSampler()
        X_train_resampled, y_train_resampled = oversampler.fit_resample(X_train, y_train)

        # use oversample data to train model
        model.fit(X_train_resampled, y_train_resampled)
        
        # calculate auc
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(fold_auc)

    # return 5-fold cv auc
    return np.mean(auc_scores)

In [6]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_1a"
    model_category = "xgboost"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 100 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2020))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=100, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 03:01:05,616] A new study created in memory with name: model_1a_tuning
[I 2026-05-11 03:01:17,339] Trial 17 finished with value: 0.8395166886979512 and parameters: {'learning_rate': 0.6699600872019681, 'max_depth': 2, 'min_child_weight': 6, 'subsample': 0.6939186395497982, 'colsample_bytree': 0.6077614185817994, 'gamma': 0.07416422361205725, 'reg_lambda': 1.0576897516720317e-07}. Best is trial 17 with value: 0.8395166886979512.
[I 2026-05-11 03:01:18,638] Trial 5 finished with value: 0.8341608834545055 and parameters: {'learning_rate': 0.8430522018575914, 'max_depth': 2, 'min_child_weight': 8, 'subsample': 0.6439487147913768, 'colsample_bytree': 0.5229779922768167, 'gamma': 3.1584874823780087e-07, 'reg_lambda': 0.121229035364304}. Best is trial 17 with value: 0.8395166886979512.
[I 2026-05-11 03:01:18,848] Trial 16 finished with value: 0.7996996401556877 and parameters: {'learning_rate': 0.8872078688884745, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.35157315411

Time: 0:02:32.689529


Model 2a:

In [7]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_2a"
    model_category = "xgboost"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 100 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2020))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=100, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 03:03:38,327] A new study created in memory with name: model_2a_tuning
[I 2026-05-11 03:03:50,450] Trial 10 finished with value: 0.7569427653815408 and parameters: {'learning_rate': 0.9920884421659479, 'max_depth': 2, 'min_child_weight': 9, 'subsample': 0.7932849193409078, 'colsample_bytree': 0.5117050063891507, 'gamma': 3.8159063278546265e-06, 'reg_lambda': 0.0011759607056977025}. Best is trial 10 with value: 0.7569427653815408.
[I 2026-05-11 03:03:50,671] Trial 11 finished with value: 0.7512190230425088 and parameters: {'learning_rate': 0.7158604708596493, 'max_depth': 2, 'min_child_weight': 2, 'subsample': 0.4774471435669251, 'colsample_bytree': 0.8362430021600324, 'gamma': 4.5584458065893814e-07, 'reg_lambda': 0.00027696021496043853}. Best is trial 10 with value: 0.7569427653815408.
[I 2026-05-11 03:03:53,516] Trial 12 finished with value: 0.7675867775549416 and parameters: {'learning_rate': 0.2867605623132874, 'max_depth': 4, 'min_child_weight': 9, 'subsample': 0.742

Time: 0:01:47.750986


Model 1b:

In [8]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_1b"
    model_category = "xgboost"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 100 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2020))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=100, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 03:05:26,117] A new study created in memory with name: model_1b_tuning
[I 2026-05-11 03:05:42,071] Trial 3 finished with value: 0.8621112254981977 and parameters: {'learning_rate': 0.9474079525219902, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.9771541394586635, 'colsample_bytree': 0.5342115129070942, 'gamma': 2.9542315122645053e-07, 'reg_lambda': 0.011715529651712669}. Best is trial 3 with value: 0.8621112254981977.
[I 2026-05-11 03:05:43,142] Trial 12 finished with value: 0.8686741993214954 and parameters: {'learning_rate': 0.06205873698581507, 'max_depth': 2, 'min_child_weight': 10, 'subsample': 0.48766428301466525, 'colsample_bytree': 0.561175762943737, 'gamma': 0.0009770843019108952, 'reg_lambda': 0.0007402958550595036}. Best is trial 12 with value: 0.8686741993214954.
[I 2026-05-11 03:05:44,656] Trial 18 finished with value: 0.887480840817765 and parameters: {'learning_rate': 0.10013524247762097, 'max_depth': 4, 'min_child_weight': 6, 'subsample': 0.882725

Time: 0:02:34.945066


Model 2b:

In [9]:
if __name__ == "__main__":
    start_time = datetime.now()
    
    # set data
    model_id = "model_2b"
    model_category = "xgboost"
    
    # load data
    X_data = pd.read_csv(f"data/X_train_{model_id}.csv", keep_default_na=False)
    y_data = pd.read_csv(f"data/y_train_{model_id}.csv", keep_default_na=False).values.ravel()

    # run 100 trails
    study = optuna.create_study(direction="maximize", study_name=f"{model_id}_tuning", sampler=optuna.samplers.TPESampler(seed=2020))
    study.optimize(lambda trial: objective_a(trial, X_data, y_data), n_trials=100, n_jobs=-1)
    # save as fig
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
    save_path = f"figures/{model_id}_{model_category}_param_importance.png"
    fig.write_image(save_path)
    print(f"Time: {datetime.now() - start_time}")

[I 2026-05-11 03:08:01,075] A new study created in memory with name: model_2b_tuning
[I 2026-05-11 03:08:17,663] Trial 4 finished with value: 0.8336380609933276 and parameters: {'learning_rate': 0.12389177436757864, 'max_depth': 2, 'min_child_weight': 4, 'subsample': 0.9778756463900459, 'colsample_bytree': 0.8380295313981474, 'gamma': 0.0002925876757481241, 'reg_lambda': 0.0009256235815031961}. Best is trial 4 with value: 0.8336380609933276.
[I 2026-05-11 03:08:18,214] Trial 19 finished with value: 0.8308365947710377 and parameters: {'learning_rate': 0.12326475474990495, 'max_depth': 2, 'min_child_weight': 4, 'subsample': 0.46967692829807983, 'colsample_bytree': 0.5545145395137614, 'gamma': 0.08730595317447888, 'reg_lambda': 0.0026985600887626414}. Best is trial 4 with value: 0.8336380609933276.
[I 2026-05-11 03:08:18,656] Trial 12 finished with value: 0.827259246459295 and parameters: {'learning_rate': 0.5405050426902621, 'max_depth': 3, 'min_child_weight': 6, 'subsample': 0.588446484

Time: 0:02:12.485599
